In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# ===============================================================
# Load and preprocess data
# ===============================================================
df = pd.read_csv('../../datasets/group_14.csv')
df["focus_factor"] = (
    df["focus_factor"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)
df["focus_factor"] = pd.to_numeric(df["focus_factor"], errors="coerce").astype(float)

data_set = df.copy()
y = data_set['target_class']
X = data_set.drop(columns=["target_class"])

# ===============================================================
# Bootstrap Evaluation Function
# ===============================================================
def evaluate_with_bootstrap(model, X, y, n_bootstrap=100, scale=True, verbose=True):
    X_np = np.array(X)
    y_np = np.array(y)

    if scale:
        scaler = StandardScaler()
        X_np = scaler.fit_transform(X_np)

    n_samples = len(y_np)
    acc_scores, f1_scores = [], []

    for b in range(n_bootstrap):
        # Sample with replacement
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        oob_indices = np.setdiff1d(np.arange(n_samples), indices)

        if len(oob_indices) == 0:
            continue  # skip if no OOB samples

        X_train, y_train = X_np[indices], y_np[indices]
        X_test, y_test = X_np[oob_indices], y_np[oob_indices]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc_scores.append(accuracy_score(y_test, y_pred))
        f1_scores.append(f1_score(y_test, y_pred, average="macro"))

    acc_mean, acc_std = np.mean(acc_scores), np.std(acc_scores)
    f1_mean, f1_std = np.mean(f1_scores), np.std(f1_scores)

    if verbose:
        print("\n" + "=" * 60)
        print(f"Model: {model.__class__.__name__} (Bootstrap Evaluation)")
        print("=" * 60)
        print(f"Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
        print(f"Macro F1-score: {f1_mean:.4f} ± {f1_std:.4f}")

    return {
        "model": model.__class__.__name__,
        "accuracy": acc_mean,
        "accuracy_std": acc_std,
        "macro_f1": f1_mean,
        "macro_f1_std": f1_std
    }

# ===============================================================
# Logistic Regression
# ===============================================================
logreg_result = evaluate_with_bootstrap(
    LogisticRegression(multi_class='ovr', solver='liblinear', max_iter=1000),
    X, y
)

# ===============================================================
# LDA
# ===============================================================
lda_result = evaluate_with_bootstrap(LDA(), X, y)

# ===============================================================
# QDA with regularization search
# ===============================================================
reg_params = np.linspace(0.1, 1.0, 5)
qda_results_list = []

for reg in reg_params:
    result = evaluate_with_bootstrap(QDA(reg_param=reg), X, y, verbose=False)
    qda_results_list.append({
        "reg_param": reg,
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"]
    })

qda_results_df = pd.DataFrame(qda_results_list)
best_idx = qda_results_df["macro_f1"].idxmax()
best_reg = qda_results_df.loc[best_idx, "reg_param"]
best_macro_f1 = qda_results_df.loc[best_idx, "macro_f1"]

print(f"\nBest QDA reg_param: {best_reg:.2f} with Macro F1-score: {best_macro_f1:.4f}")

# ----------------------------------------------------------
# Plot QDA performance vs reg_param
# ----------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.plot(qda_results_df["reg_param"], qda_results_df["accuracy"], marker='o', label="Accuracy")
plt.plot(qda_results_df["reg_param"], qda_results_df["macro_f1"], marker='s', linestyle='--', label="Macro F1")
plt.scatter(best_reg, best_macro_f1, color='red', s=100, label=f"Best reg_param = {best_reg:.2f}")
plt.title("QDA Performance vs Regularization Parameter (Bootstrap)")
plt.xlabel("reg_param")
plt.ylabel("Score")
plt.xticks(reg_params)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

# ===============================================================
# Best QDA final evaluation
# ===============================================================
qda_best_result = evaluate_with_bootstrap(QDA(reg_param=best_reg), X, y)

# ===============================================================
# Compare model accuracies
# ===============================================================
accuracies = {
    "Logistic Regression": logreg_result["accuracy"],
    "LDA": lda_result["accuracy"],
    "QDA (best reg_param)": qda_best_result["accuracy"]
}

plt.figure(figsize=(6, 4))
sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()))
plt.title("Model Accuracy Comparison (Bootstrap)")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()